# 01 · Data Collection & Preprocessing  ·  *Role A (Data Engineer)*

**Goal (Part 1, data half):** turn raw xBD tiles into a clean, patch-level dataset
`(pre_patch, post_patch, damage_label, disaster_type)` with **per-disaster splits**.

Dataset: xView2 / xBD — https://xview2.org/  (request access, download train+tier3).
Alternatives: Maxar Open Data, Copernicus EMS, Sentinel-2 (coarser).

### Expected raw layout
```
data/raw/xbd/
  images/  <event>_<id>_pre_disaster.png   <event>_<id>_post_disaster.png
  labels/  <event>_<id>_pre_disaster.json  <event>_<id>_post_disaster.json
```
Each post-disaster label JSON has building polygons with a `subtype` =
`no-damage | minor-damage | major-damage | destroyed | un-classified`.

In [ ]:
import sys, os; sys.path.append(os.path.abspath('..'))
from src import config, data_utils
import json, glob
import numpy as np
from pathlib import Path

RAW = Path(config.DATA_DIR) / 'raw' / 'xbd'
print('Looking for raw data in:', RAW.resolve())
print('Exists:', RAW.exists())

### Step 1 — Parse a label JSON
TODO: read a post-disaster JSON, extract building polygons (WKT) + damage subtype.
Map the disaster name from the filename to a `disaster_type`.

In [ ]:
# TODO(Role A): implement label parsing.
def parse_label(json_path):
    """Return list of dicts: {polygon, subtype, disaster_type}."""
    with open(json_path) as f:
        data = json.load(f)
    buildings = []
    for feat in data['features'].get('xy', []):
        subtype = feat['properties'].get('subtype', 'un-classified')
        buildings.append({'wkt': feat['wkt'], 'subtype': subtype})
    # disaster_type is the leading token of the event name, e.g. 'hurricane-michael'
    disaster = Path(json_path).stem.split('_')[0]
    return buildings, disaster

# Example (uncomment once data is present):
# sample = next(iter(glob.glob(str(RAW/'labels'/'*post*.json'))), None)
# print(parse_label(sample))

### Step 2 — Co-register & crop building patches
Pre and post xBD tiles are already co-registered (same grid). For each building polygon:
compute its bounding box, pad it, and crop the **same window** from both pre and post images.
Resize every patch to `config.PATCH_SIZE`.

TODO: handle edge buildings, skip `un-classified`, optionally skip tiny polygons.

In [ ]:
# TODO(Role A): crop one (pre, post) patch pair for a building polygon.
from shapely import wkt as shapely_wkt  # pip install shapely
from PIL import Image

def crop_patch(img, bbox, size=config.PATCH_SIZE, pad=10):
    minx, miny, maxx, maxy = bbox
    crop = img.crop((minx-pad, miny-pad, maxx+pad, maxy+pad))
    return crop.resize((size, size))

def building_bbox(wkt_str):
    geom = shapely_wkt.loads(wkt_str)
    minx, miny, maxx, maxy = geom.bounds
    return int(minx), int(miny), int(maxx), int(maxy)

### Step 3 — Quality cleaning
Remove cloud / smoke-obscured / corrupted tiles, standardize resolution & bands, normalize.
TODO: simple heuristics (e.g. drop near-white/near-black patches, brightness thresholds).

In [ ]:
# TODO(Role A): quality filters.
def is_bad_patch(patch_np, white_frac=0.5):
    gray = patch_np.mean(axis=-1)
    too_white = (gray > 240).mean() > white_frac
    too_dark  = (gray < 15).mean()  > white_frac
    return too_white or too_dark

### Step 4 — Build the patch index & per-disaster split
Write a manifest CSV: `pre_path, post_path, label, disaster_type, split`.
**Split rule (critical for grading):** train/val on `config.TRAIN_DISASTERS`,
test entirely on `config.HELDOUT_DISASTER` — *not* a random split.

In [ ]:
# TODO(Role A): iterate all tiles -> crop patches -> save -> build manifest.
import pandas as pd

PROC = Path(config.DATA_DIR) / 'processed'
(PROC / 'patches').mkdir(parents=True, exist_ok=True)

def assign_split(disaster_type):
    if disaster_type == config.HELDOUT_DISASTER:
        return 'test'
    return 'train'  # carve out a val fraction later, stratified by class

# rows = [...]; manifest = pd.DataFrame(rows)
# manifest.to_csv(PROC / 'manifest.csv', index=False)
# print(manifest['split'].value_counts())

### Step 5 — Class-imbalance strategy
Most buildings are **no-damage**. Decide and *justify*:
class weights in the loss, oversampling rare classes, or focal loss.
Compute class counts here and store `class_weights` in the manifest folder for the model notebooks.

In [ ]:
# TODO(Role A): compute class weights from the train split.
# counts = manifest[manifest.split=='train']['label'].value_counts().sort_index()
# weights = counts.sum() / (len(counts) * counts)
# np.save(PROC / 'class_weights.npy', weights.values)
# print(counts); print(weights)

**Deliverable for B/C/D:** `data/processed/manifest.csv` + `class_weights.npy`.
Once this exists, `data_utils.DamageDataset` (see `src/data_utils.py`) loads it directly.